# RAG Chatbot — LightRAG + Agno
Ingest PDF/Excel documents into a knowledge graph, then chat with them via a Gemini-powered agent.

In [ ]:
import os
import asyncio
import nest_asyncio
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()

# Agno's Gemini model reads GOOGLE_API_KEY (agno/models/google/gemini.py:149)
if os.getenv("GEMINI_API_KEY") and not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

print("Environment ready")

In [ ]:
import time as _time
from sentence_transformers import SentenceTransformer
from mistralai import Mistral, UserMessage, SystemMessage, AssistantMessage
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc


# --- Per-call rate limiter (token-bucket style) ---
class RateLimiter:
    """Enforces a minimum delay between API calls across all concurrent workers."""
    def __init__(self, requests_per_minute: int = 30):
        self._delay = 60.0 / requests_per_minute
        self._lock = asyncio.Lock()
        self._last_call = 0.0

    async def wait(self):
        async with self._lock:
            now = _time.monotonic()
            wait_time = self._delay - (now - self._last_call)
            if wait_time > 0:
                await asyncio.sleep(wait_time)
            self._last_call = _time.monotonic()

rate_limiter = RateLimiter(requests_per_minute=30)   # ← tune this


# --- Local embeddings (384d, no API cost) ---
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

async def embedding_func(texts: list[str]) -> np.ndarray:
    return embed_model.encode(texts, normalize_embeddings=True)


# --- Mistral LLM for LightRAG internals ---
mistral_client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

async def llm_model_func(
    prompt, system_prompt=None, history_messages=None, keyword_extraction=False, **kwargs
) -> str:
    messages = []
    if system_prompt:
        messages.append(SystemMessage(content=system_prompt))
    if history_messages:
        for msg in history_messages:
            if msg["role"] == "assistant":
                messages.append(AssistantMessage(content=msg["content"]))
            else:
                messages.append(UserMessage(content=msg["content"]))
    messages.append(UserMessage(content=prompt))

    await rate_limiter.wait()
    response = await mistral_client.chat.complete_async(
        model="ministral-14b-2512",
        messages=messages,
    )
    return response.choices[0].message.content


# --- Init LightRAG ---
WORKING_DIR = "./rag_storage"
os.makedirs(WORKING_DIR, exist_ok=True)

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=llm_model_func,
    llm_model_max_async=4,          # concurrent workers (default)
    embedding_func=EmbeddingFunc(
        embedding_dim=384,
        max_token_size=8192,
        func=embedding_func,
    ),
    chunk_token_size=1200,
    chunk_overlap_token_size=100,
)
await rag.initialize_storages()
print(f"LightRAG initialized — rate limit: {rate_limiter._delay:.1f}s between calls, max_async: 4")

## Document Indexing
Add file paths below and run to parse + index into the knowledge graph.

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

async def process_and_index(file_paths: list[str]):
    for path in file_paths:
        print(f"Parsing {path}...")
        result = converter.convert(path)
        text = result.document.export_to_markdown()
        if not text.strip():
            print(f"  Skipped (no text extracted)")
            continue
        print(f"  Extracted {len(text)} chars, indexing...")
        await rag.ainsert(text)
        print(f"  Done: {Path(path).name}")

# --- Edit this list with your files ---
files_to_index = [
    'sample_docs/sample_statement.pdf'
    # "/path/to/spreadsheet.xlsx",
]

if files_to_index:
    await process_and_index(files_to_index)
else:
    print("No files to index. Add paths to files_to_index and re-run.")

## Chat
Run the cells below to start an interactive chat session. Type `quit` or `exit` to stop.

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools import Toolkit


class RAGToolkit(Toolkit):
    def __init__(self):
        super().__init__(name="rag_tools")
        self.register(self.search_documents)

    async def search_documents(self, query: str) -> str:
        """Search the indexed documents for information relevant to the query."""
        return await rag.aquery(query, param=QueryParam(mode="hybrid", top_k=60))


agent = Agent(
    name="RAG Chatbot",
    model=Gemini(id="gemini-2.5-flash"),
    tools=[RAGToolkit()],
    instructions=[
        "Always use search_documents before answering a question.",
        "Ground your answers in the search results. If the documents don't contain the answer, say so.",
    ],
    add_history_to_context=True,
    num_history_runs=5,
    markdown=True,
)
print("Agent ready")

In [ ]:
print("Chat started. Type 'quit' or 'exit' to stop.\n")

try:
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if not user_input.strip():
            continue
        response = await agent.arun(user_input)
        print(f"\nAssistant: {response.content}\n")
except KeyboardInterrupt:
    print("\nChat stopped.")